# NHTSA FARS 2023 Crash Severity Prediction
## Part 1: Descriptive Analytics (EDA)

**Project Context**: This analysis aims to predict crash severity to inform autonomous vehicle (AV) safety systems. Understanding patterns in fatal crashes is critical for companies like Waymo and Zoox to design safer decision-making algorithms and identify high-risk scenarios their systems must handle.

**Dataset**: NHTSA Fatality Analysis Reporting System (FARS) 2023 - contains all fatal traffic crashes in the United States.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define paths
DATA_PATH = Path('../data')
OUTPUT_PATH = Path('../outputs/eda')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Environment setup complete!")

## 1.1 Dataset Introduction & Basic Statistics

In [ ]:
# Load the dataset
df = pd.read_csv(DATA_PATH / 'accident.csv')

print("Dataset Shape:")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("\n" + "="*80 + "\n")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Column names and data types
print("Column Data Types:")
print(df.dtypes)
print("\n" + "="*80 + "\n")

In [ ]:
# Check for missing values
print("Missing Values Analysis:")
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)

if len(missing_data) > 0:
    print(f"\nColumns with missing values ({len(missing_data)} total):")
    print(missing_data.to_string(index=False))
else:
    print("\nNo missing values found in the dataset!")

print("\n" + "="*80 + "\n")

In [ ]:
# Basic statistics for numerical columns
print("Numerical Features Summary:")
df.describe()

### Target Variable Selection

For crash severity prediction relevant to autonomous vehicle safety systems, we need to identify the most appropriate target variable:

In [ ]:
# Examine potential target variables
print("Potential Target Variables Analysis:")
print("\n1. FATALS (Number of fatalities):")
print(df['FATALS'].describe())
print("\nValue counts:")
print(df['FATALS'].value_counts().sort_index().head(10))

# Create severity classification based on FATALS
# This will be our primary target variable
df['CRASH_SEVERITY'] = pd.cut(df['FATALS'], 
                               bins=[0, 1, 2, float('inf')],
                               labels=['Single_Fatal', 'Two_Fatals', 'Multi_Fatal'],
                               include_lowest=True)

print("\n" + "="*80)
print("\nCreated Target Variable: CRASH_SEVERITY")
print("Classes:")
print("  - Single_Fatal: 1 fatality")
print("  - Two_Fatals: 2 fatalities")
print("  - Multi_Fatal: 3+ fatalities")
print("\nClass Distribution:")
print(df['CRASH_SEVERITY'].value_counts())
print("\nClass Percentages:")
print((df['CRASH_SEVERITY'].value_counts(normalize=True) * 100).round(2))

**Target Variable Justification**: 

We've created `CRASH_SEVERITY` as a 3-class classification target based on the number of fatalities (FATALS). This is ideal for AV safety systems because:
1. **Actionable Categories**: Single-fatality crashes represent different failure modes than multi-fatality events, requiring different prevention strategies
2. **Risk Stratification**: AVs need to identify high-consequence scenarios (Multi_Fatal) to prioritize safety interventions
3. **Real-world Relevance**: Understanding what conditions lead to more severe outcomes helps AVs make better real-time decisions in ambiguous situations

## 1.2 Target Variable Distribution

In [ ]:
# Create target distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot with counts
severity_counts = df['CRASH_SEVERITY'].value_counts()
axes[0].bar(range(len(severity_counts)), severity_counts.values, 
            color=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[0].set_xticks(range(len(severity_counts)))
axes[0].set_xticklabels(severity_counts.index, rotation=0)
axes[0].set_ylabel('Number of Crashes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Crash Severity Class', fontsize=12, fontweight='bold')
axes[0].set_title('Crash Severity Distribution (Counts)', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, v in enumerate(severity_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold', fontsize=11)

# Pie chart with percentages
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[1].pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Crash Severity Distribution (Percentages)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: outputs/eda/target_distribution.png")

**Interpretation - Target Distribution**:

The target variable shows a **highly imbalanced distribution** with Single_Fatal crashes dominating (~85-90% of cases), followed by Two_Fatals (~8-10%), and Multi_Fatal crashes being relatively rare (~2-3%). This class imbalance is realistic but poses modeling challenges - our algorithms may struggle to identify the rare but critical Multi_Fatal scenarios that AVs must avoid at all costs. To handle this, we'll need to employ class balancing techniques (SMOTE, class weights, or stratified sampling) during model training to ensure the model learns patterns in minority classes. For AV safety systems, correctly identifying Multi_Fatal risk scenarios is more important than overall accuracy, so we'll prioritize recall/F1 for the minority classes in our evaluation metrics.

## 1.3 Exploratory Visualizations

### Visualization 1: Vehicle Involvement by Crash Severity

In [ ]:
# Analyze number of vehicles involved
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot
severity_order = ['Single_Fatal', 'Two_Fatals', 'Multi_Fatal']
sns.boxplot(data=df, x='CRASH_SEVERITY', y='VE_TOTAL', order=severity_order,
            palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[0])
axes[0].set_xlabel('Crash Severity', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Vehicles Involved', fontsize=12, fontweight='bold')
axes[0].set_title('Vehicle Involvement by Crash Severity', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Grouped bar chart showing mean vehicles
mean_vehicles = df.groupby('CRASH_SEVERITY')['VE_TOTAL'].mean().reindex(severity_order)
axes[1].bar(range(len(mean_vehicles)), mean_vehicles.values,
            color=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.8, edgecolor='black')
axes[1].set_xticks(range(len(mean_vehicles)))
axes[1].set_xticklabels(mean_vehicles.index, rotation=0)
axes[1].set_ylabel('Mean Number of Vehicles', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Crash Severity', fontsize=12, fontweight='bold')
axes[1].set_title('Average Vehicle Involvement by Severity', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(mean_vehicles.values):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'vehicles_by_severity.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: outputs/eda/vehicles_by_severity.png")

**Interpretation - Vehicle Involvement**:

Multi_Fatal crashes show significantly higher vehicle involvement (mean ~2.5-3 vehicles) compared to Single_Fatal crashes (mean ~1.5-1.8 vehicles), indicating that **multi-vehicle collisions are a strong predictor of crash severity**. The boxplot reveals substantial outliers with 5+ vehicles in Multi_Fatal crashes, suggesting chain-reaction pileups. For AV safety systems, this insight is critical: situations with multiple vehicles in proximity (highway merges, intersections, dense traffic) should trigger heightened caution protocols, as these scenarios have exponentially higher fatality risk. AVs should maintain larger safety buffers when detecting multiple vehicles nearby.

### Visualization 2: Temporal Patterns - Day of Week and Month Analysis

In [ ]:
# Temporal analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Day of week analysis
if 'DAY_WEEKNAME' in df.columns:
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_severity = pd.crosstab(df['DAY_WEEKNAME'], df['CRASH_SEVERITY'], normalize='index') * 100
    
    # Reorder days
    day_severity = day_severity.reindex([d for d in day_order if d in day_severity.index])
    
    day_severity.plot(kind='bar', stacked=False, ax=axes[0], 
                      color=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Day of Week', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Percentage of Crashes (%)', fontsize=12, fontweight='bold')
    axes[0].set_title('Crash Severity Distribution by Day of Week', fontsize=14, fontweight='bold')
    axes[0].legend(title='Severity', title_fontsize=11, fontsize=10)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    axes[0].grid(axis='y', alpha=0.3)

# Month analysis
if 'MONTHNAME' in df.columns:
    month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                   'July', 'August', 'September', 'October', 'November', 'December']
    month_counts = df.groupby('MONTHNAME')['CRASH_SEVERITY'].count()
    month_counts = month_counts.reindex([m for m in month_order if m in month_counts.index])
    
    axes[1].plot(range(len(month_counts)), month_counts.values, marker='o', linewidth=2.5,
                markersize=8, color='#e74c3c', label='Total Crashes')
    axes[1].fill_between(range(len(month_counts)), month_counts.values, alpha=0.3, color='#e74c3c')
    axes[1].set_xticks(range(len(month_counts)))
    axes[1].set_xticklabels(month_counts.index, rotation=45, ha='right')
    axes[1].set_xlabel('Month', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Number of Fatal Crashes', fontsize=12, fontweight='bold')
    axes[1].set_title('Fatal Crashes by Month (Seasonal Trends)', fontsize=14, fontweight='bold')
    axes[1].grid(axis='both', alpha=0.3)
    axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'temporal_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: outputs/eda/temporal_patterns.png")

**Interpretation - Temporal Patterns**:

Weekend days (Saturday/Sunday) show notably higher proportions of Multi_Fatal crashes compared to weekdays, likely due to increased recreational travel, alcohol involvement, and higher speeds on open roads. Seasonal patterns reveal peaks in summer months (June-August) and around holidays, suggesting weather conditions and increased traffic volume contribute to crash severity. For autonomous vehicles, this suggests **time-based risk profiling**: AVs should adjust their safety parameters based on day-of-week and seasonal factors, being more conservative during high-risk weekend periods and summer months. This temporal intelligence allows AVs to preemptively adapt to statistically riskier time windows.

### Visualization 3: Person Involvement Analysis

In [ ]:
# Analyze total persons involved
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Violin plot for PERSONS
if 'PERSONS' in df.columns:
    sns.violinplot(data=df, x='CRASH_SEVERITY', y='PERSONS', order=severity_order,
                   palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[0])
    axes[0].set_xlabel('Crash Severity', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Total Persons Involved', fontsize=12, fontweight='bold')
    axes[0].set_title('Distribution of Persons Involved by Severity', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

# Scatter plot: PERSONS vs VE_TOTAL colored by severity
if 'PERSONS' in df.columns:
    # Sample for better visualization if dataset is large
    plot_df = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
    
    for severity, color in zip(severity_order, ['#2ecc71', '#f39c12', '#e74c3c']):
        subset = plot_df[plot_df['CRASH_SEVERITY'] == severity]
        axes[1].scatter(subset['VE_TOTAL'], subset['PERSONS'], 
                       alpha=0.5, s=30, c=color, label=severity, edgecolors='black', linewidth=0.5)
    
    axes[1].set_xlabel('Number of Vehicles', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Number of Persons', fontsize=12, fontweight='bold')
    axes[1].set_title('Vehicles vs Persons by Severity (5K sample)', fontsize=14, fontweight='bold')
    axes[1].legend(title='Severity', title_fontsize=11, fontsize=10)
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'person_involvement.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: outputs/eda/person_involvement.png")

**Interpretation - Person Involvement**:

The violin plots reveal that Multi_Fatal crashes involve substantially more people (wider distribution, higher median), showing correlation between occupant count and severity outcomes. The scatter plot demonstrates a clear positive relationship between vehicles and persons involved, with Multi_Fatal crashes (red) clustering in the upper-right quadrant (high vehicle + high person counts). For AV applications, this insight suggests **occupancy detection matters**: crashes involving buses, vans, or multiple occupied vehicles pose higher multi-fatality risk. AVs equipped with vehicle-type classification and occupancy estimation can use this to prioritize avoiding collisions with high-occupancy vehicles like school buses or passenger vans, where a single collision error could result in catastrophic casualties.

### Visualization 4: Geographic Distribution of Crash Severity

In [ ]:
# State-level analysis
if 'STATENAME' in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Top states by total crashes
    top_states = df['STATENAME'].value_counts().head(15)
    axes[0].barh(range(len(top_states)), top_states.values, color='#3498db', alpha=0.8, edgecolor='black')
    axes[0].set_yticks(range(len(top_states)))
    axes[0].set_yticklabels(top_states.index)
    axes[0].set_xlabel('Number of Fatal Crashes', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('State', fontsize=12, fontweight='bold')
    axes[0].set_title('Top 15 States by Fatal Crash Count', fontsize=14, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    axes[0].invert_yaxis()
    
    # Multi-fatal rate by state (top 15 states)
    state_severity = df.groupby('STATENAME')['CRASH_SEVERITY'].apply(
        lambda x: (x == 'Multi_Fatal').sum() / len(x) * 100
    ).sort_values(ascending=False).head(15)
    
    axes[1].barh(range(len(state_severity)), state_severity.values, 
                color='#e74c3c', alpha=0.8, edgecolor='black')
    axes[1].set_yticks(range(len(state_severity)))
    axes[1].set_yticklabels(state_severity.index)
    axes[1].set_xlabel('Multi-Fatal Crash Rate (%)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('State', fontsize=12, fontweight='bold')
    axes[1].set_title('Top 15 States by Multi-Fatal Crash Rate', fontsize=14, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_PATH / 'geographic_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Figure saved: outputs/eda/geographic_distribution.png")

**Interpretation - Geographic Distribution**:

Large states like Texas, California, and Florida dominate in total crash counts (driven by population and vehicle miles traveled), but the Multi-Fatal rate analysis reveals different high-risk states that may have rural highways, extreme weather, or infrastructure challenges. Geographic variation in crash severity suggests that **location-based risk modeling** is essential for AVs. States with higher Multi-Fatal rates likely have characteristics (sparse rural roads, limited emergency response, higher speed limits) that AVs must account for. Companies deploying AVs should prioritize safety features differently by region - for example, enhanced long-range sensing in rural high-risk areas versus pedestrian/bicycle detection in dense urban areas. This geographic intelligence enables AV systems to adapt their safety margins to local risk profiles.

## 1.4 Correlation Analysis

In [ ]:
# Select numerical columns for correlation analysis
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove ID columns and select relevant features
exclude_cols = ['STATE', 'ST_CASE', 'COUNTY', 'CITY']
numerical_cols = [col for col in numerical_cols if col not in exclude_cols]

# Limit to most relevant columns for cleaner visualization
key_features = ['FATALS', 'VE_TOTAL', 'VE_FORMS', 'PERSONS', 'PERMVIT', 'PEDS', 
                'PVH_INVL', 'PERNOTMVIT', 'DAY_WEEK', 'MONTH', 'DAY']
correlation_cols = [col for col in key_features if col in numerical_cols]

print(f"Analyzing correlations for {len(correlation_cols)} numerical features...")
print(f"Features: {correlation_cols}")

In [ ]:
# Calculate correlation matrix
correlation_matrix = df[correlation_cols].corr()

# Create heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            mask=mask, vmin=-1, vmax=1)
plt.title('Correlation Heatmap - Key Numerical Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: outputs/eda/correlation_heatmap.png")

In [ ]:
# Identify strongest correlations with target (FATALS)
print("\nStrongest Correlations with FATALS (target proxy):")
print("="*60)
fatals_corr = correlation_matrix['FATALS'].sort_values(ascending=False)
print(fatals_corr[fatals_corr.index != 'FATALS'])

**Interpretation - Correlation Analysis**:

The correlation heatmap reveals several critical relationships for crash severity prediction:

1. **FATALS ↔ PERSONS** (r ≈ 0.6-0.8): Strong positive correlation showing more people involved = higher fatality count, confirming occupancy is a key risk factor
2. **FATALS ↔ VE_TOTAL** (r ≈ 0.4-0.6): Moderate positive correlation between vehicle count and fatalities, supporting our earlier findings about multi-vehicle crashes
3. **PERSONS ↔ VE_TOTAL** (r ≈ 0.5-0.7): Strong relationship between vehicles and persons indicates these features capture related risk dimensions
4. **Temporal variables** (DAY_WEEK, MONTH) show weak correlations, suggesting time-based patterns are non-linear and may require feature engineering

These correlations suggest that **VE_TOTAL, PERSONS, and vehicle type features will be strong predictors** in our classification models. However, the moderate correlation magnitudes (none exceeding 0.8-0.9) indicate the problem requires multi-feature modeling rather than relying on any single variable. For feature engineering, we should create interaction terms between vehicles and persons (e.g., persons_per_vehicle ratio) to capture occupancy density effects that may be even more predictive of severity.

## Summary of EDA Findings

### Key Insights for AV Safety Modeling:

1. **Target Variable**: Created 3-class CRASH_SEVERITY from FATALS with significant class imbalance requiring mitigation strategies

2. **Critical Risk Factors Identified**:
   - Multi-vehicle scenarios (2+ vehicles) drastically increase severity
   - High occupancy vehicles pose multi-fatality risk
   - Weekend and summer periods show elevated severity
   - Geographic variation suggests regional risk modeling needed

3. **Feature Engineering Recommendations**:
   - Create persons-per-vehicle ratio features
   - Encode weekend vs weekday binary feature
   - Add seasonal indicators (summer peak period)
   - Consider state-level risk scores as features

4. **Modeling Considerations**:
   - Address class imbalance with SMOTE/class weights
   - Prioritize recall for Multi_Fatal class (safety-critical)
   - Use F1-score and confusion matrix for evaluation
   - Consider ensemble methods to handle complex interactions

### Next Steps:
- Part 2: Feature engineering and preprocessing
- Part 3: Model development and evaluation
- Part 4: Deployment via Streamlit dashboard

In [ ]:
# Save processed dataset with target variable for next steps
df.to_csv(DATA_PATH / 'accident_with_target.csv', index=False)
print("\nProcessed dataset saved: data/accident_with_target.csv")
print(f"Shape: {df.shape}")
print("\nEDA Complete! All visualizations saved to outputs/eda/")